In [21]:
# ============================================================
# DAY 1-2: IFRS S1/S2 RAG KNOWLEDGE BASE
# ============================================================

import os
import re
import json
import psycopg2
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple
from dataclasses import dataclass
import pandas as pd 

# PDF parsing
import pypdf

# Embeddings — use sentence-transformers locally (free, no API cost)
from sentence_transformers import SentenceTransformer

# Or use OpenAI embeddings if you have a key
# from openai import OpenAI

print("Dependencies loaded")

Dependencies loaded


In [14]:
# ── DATABASE CONNECTION ──────────────────────────────────────
DB_CONFIG = {
    "host":     "localhost",
    "port":     5432,
    "dbname":   "ESG_Data",
    "user":     "postgres",
    "password": "admin"
}

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

# Test connection
conn = get_conn()
print("Database connected")
conn.close()

Database connected


In [24]:
import pymupdf4llm

md_text = pymupdf4llm.to_markdown("gen_data/ifrs_s1.pdf")

=== Document parser messages ===
Using Tesseract for OCR processing.



In [25]:
import re
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class IFRSParagraph:
    para_id: str          # "27", "B13", "D4", "C2"
    section: str          # "core_content", "appendix_b", etc.
    is_bold: bool         # True = normative principle
    text: str
    sub_items: list[str] = field(default_factory=list)
    raw_cross_refs: list[str] = field(default_factory=list)

PARA_PATTERN = re.compile(
    r'^(\*\*)?(B\d+|C\d+|D\d+|E\d+|\d+)(\*\*)?\s+(.+)',
    re.MULTILINE
)

XREF_PATTERN = re.compile(
    r'paragraph[s]?\s+([\w\d\(\)–\-,\s]+)',
    re.IGNORECASE
)

def parse_ifrs_paragraphs(md_text: str) -> list[IFRSParagraph]:
    paragraphs = []
    
    for match in PARA_PATTERN.finditer(md_text):
        bold_open = match.group(1)
        para_num = match.group(2)
        text = match.group(4).strip()
        
        is_bold = bold_open == "**"
        section = classify_section(para_num)
        cross_refs = XREF_PATTERN.findall(text)
        
        paragraphs.append(IFRSParagraph(
            para_id=para_num,
            section=section,
            is_bold=is_bold,
            text=text,
            raw_cross_refs=cross_refs,
        ))
    
    return paragraphs

def classify_section(para_id: str) -> str:
    if para_id.startswith("B"): return "appendix_b"
    if para_id.startswith("C"): return "appendix_c"
    if para_id.startswith("D"): return "appendix_d"
    if para_id.startswith("E"): return "appendix_e"
    num = int(para_id)
    if num <= 4:   return "objective"
    if num <= 9:   return "scope"
    if num <= 24:  return "conceptual_foundations"
    if num <= 53:  return "core_content"
    if num <= 73:  return "general_requirements"
    return "judgements_and_errors"

In [26]:
# Explicit semantic chunk definitions
# Each entry: (chunk_id, display_name, para_ids, paired_appendix_ids, cross_std_refs)

IFRS_S1_CHUNK_MAP = [
    (
        "s1_objective_scope",
        "Objective and Scope",
        list(range(1, 10)),          # §1–9
        [],
        [],
    ),
    (
        "s1_fair_presentation",
        "Fair Presentation",
        list(range(11, 17)),         # §11–16
        [],
        [],
    ),
    (
        "s1_materiality",
        "Materiality",
        list(range(17, 20)),         # §17–19
        [f"B{i}" for i in range(13, 38)],  # B13–B37
        [],
    ),
    (
        "s1_connected_information",
        "Connected Information",
        list(range(21, 25)),         # §21–24
        [f"B{i}" for i in range(39, 45)],  # B39–B44
        [],
    ),
    (
        "s1_governance",
        "Governance — Core Content",
        [26, 27],
        [],
        ["IFRS_S2_5", "IFRS_S2_6", "IFRS_S2_7"],
    ),
    (
        "s1_strategy_risks",
        "Strategy — Risks and Opportunities",
        list(range(28, 32)),         # §28–31
        [],
        [],
    ),
    (
        "s1_strategy_financial_effects",
        "Strategy — Financial Effects",
        list(range(34, 41)),         # §34–40
        [],
        [],
    ),
    (
        "s1_risk_management",
        "Risk Management",
        [43, 44],
        [],
        [],
    ),
    (
        "s1_metrics_and_targets",
        "Metrics and Targets",
        list(range(45, 54)),         # §45–53
        [f"B{i}" for i in range(49, 55)],  # B49–B54
        [],
    ),
    (
        "s1_sources_of_guidance",
        "Sources of Guidance",
        list(range(54, 60)),         # §54–59
        [],
        [],
    ),
    (
        "s1_measurement_uncertainty",
        "Measurement Uncertainty",
        list(range(77, 83)),         # §77–82
        [],
        [],
    ),
    (
        "s1_errors",
        "Errors",
        list(range(83, 87)),         # §83–86
        [f"B{i}" for i in range(55, 60)],  # B55–B59
        [],
    ),
    (
        "s1_defined_terms",
        "Appendix A — Defined Terms",
        [],                          # parsed separately from table
        [],
        [],
    ),
    (
        "s1_qualitative_characteristics_fundamental",
        "Appendix D — Fundamental Qualitative Characteristics",
        [f"D{i}" for i in range(1, 16)],
        [],
        [],
    ),
    (
        "s1_qualitative_characteristics_enhancing",
        "Appendix D — Enhancing Qualitative Characteristics",
        [f"D{i}" for i in range(16, 34)],
        [],
        [],
    ),
]

In [27]:
from dataclasses import dataclass

@dataclass
class IFRSChunk:
    chunk_id: str
    display_name: str
    text: str
    metadata: dict

def build_anchor(para: IFRSParagraph) -> str:
    """Short anchor text for cross-reference injection."""
    # First 200 chars of the paragraph + ellipsis
    short = para.text[:200].rsplit(" ", 1)[0]
    return f"[Anchor §{para.para_id}]: {short}..."

def assemble_chunks(
    parsed: dict[str, IFRSParagraph],  # para_id -> IFRSParagraph
    chunk_map: list,
    anchor_xrefs: dict[str, list[str]],  # chunk_id -> [para_ids to inject as anchors]
) -> list[IFRSChunk]:

    chunks = []

    for chunk_id, display_name, para_ids, appendix_ids, cross_std_refs in chunk_map:

        all_ids = [str(p) for p in para_ids] + appendix_ids
        parts = []

        for pid in all_ids:
            if pid in parsed:
                p = parsed[pid]
                prefix = "**" if p.is_bold else ""
                parts.append(f"{prefix}§{p.para_id} {p.text}")

        # Inject cross-reference anchors
        for anchor_para_id in anchor_xrefs.get(chunk_id, []):
            if anchor_para_id in parsed:
                parts.append(build_anchor(parsed[anchor_para_id]))

        full_text = "\n\n".join(parts)

        # Collect all cross-refs found in paragraphs
        all_xrefs = []
        for pid in all_ids:
            if pid in parsed:
                all_xrefs.extend(parsed[pid].raw_cross_refs)

        chunks.append(IFRSChunk(
            chunk_id=chunk_id,
            display_name=display_name,
            text=full_text,
            metadata={
                "standard": "IFRS S1",
                "paragraph_ids": all_ids,
                "cross_std_refs": cross_std_refs,
                "internal_xrefs": list(set(all_xrefs)),
                "has_bold_principle": any(
                    parsed[str(p)].is_bold
                    for p in para_ids
                    if str(p) in parsed
                ),
                "pillar": infer_pillar(chunk_id),
            }
        ))

    return chunks


def infer_pillar(chunk_id: str) -> str:
    mapping = {
        "governance": "governance",
        "strategy": "strategy",
        "risk_management": "risk_management",
        "metrics": "metrics_and_targets",
        "materiality": "general_requirements",
        "sources": "general_requirements",
    }
    for key, pillar in mapping.items():
        if key in chunk_id:
            return pillar
    return "other"


# Cross-reference anchor injections
# chunk_id -> list of para_ids to append as short anchors
ANCHOR_INJECTIONS = {
    "s1_governance": ["51"],          # §51 target monitoring, referenced from §27(a)(v)
    "s1_strategy_risks": ["B6"],      # reasonable/supportable info definition
    "s1_metrics_and_targets": ["27"], # governance oversight of targets
    "s1_materiality": ["17"],         # main materiality principle anchor in appendix B
}

In [28]:
from dataclasses import dataclass

@dataclass
class IFRSChunk:
    chunk_id: str
    display_name: str
    text: str
    metadata: dict

def build_anchor(para: IFRSParagraph) -> str:
    """Short anchor text for cross-reference injection."""
    # First 200 chars of the paragraph + ellipsis
    short = para.text[:200].rsplit(" ", 1)[0]
    return f"[Anchor §{para.para_id}]: {short}..."

def assemble_chunks(
    parsed: dict[str, IFRSParagraph],  # para_id -> IFRSParagraph
    chunk_map: list,
    anchor_xrefs: dict[str, list[str]],  # chunk_id -> [para_ids to inject as anchors]
) -> list[IFRSChunk]:

    chunks = []

    for chunk_id, display_name, para_ids, appendix_ids, cross_std_refs in chunk_map:

        all_ids = [str(p) for p in para_ids] + appendix_ids
        parts = []

        for pid in all_ids:
            if pid in parsed:
                p = parsed[pid]
                prefix = "**" if p.is_bold else ""
                parts.append(f"{prefix}§{p.para_id} {p.text}")

        # Inject cross-reference anchors
        for anchor_para_id in anchor_xrefs.get(chunk_id, []):
            if anchor_para_id in parsed:
                parts.append(build_anchor(parsed[anchor_para_id]))

        full_text = "\n\n".join(parts)

        # Collect all cross-refs found in paragraphs
        all_xrefs = []
        for pid in all_ids:
            if pid in parsed:
                all_xrefs.extend(parsed[pid].raw_cross_refs)

        chunks.append(IFRSChunk(
            chunk_id=chunk_id,
            display_name=display_name,
            text=full_text,
            metadata={
                "standard": "IFRS S1",
                "paragraph_ids": all_ids,
                "cross_std_refs": cross_std_refs,
                "internal_xrefs": list(set(all_xrefs)),
                "has_bold_principle": any(
                    parsed[str(p)].is_bold
                    for p in para_ids
                    if str(p) in parsed
                ),
                "pillar": infer_pillar(chunk_id),
            }
        ))

    return chunks


def infer_pillar(chunk_id: str) -> str:
    mapping = {
        "governance": "governance",
        "strategy": "strategy",
        "risk_management": "risk_management",
        "metrics": "metrics_and_targets",
        "materiality": "general_requirements",
        "sources": "general_requirements",
    }
    for key, pillar in mapping.items():
        if key in chunk_id:
            return pillar
    return "other"


# Cross-reference anchor injections
# chunk_id -> list of para_ids to append as short anchors
ANCHOR_INJECTIONS = {
    "s1_governance": ["51"],          # §51 target monitoring, referenced from §27(a)(v)
    "s1_strategy_risks": ["B6"],      # reasonable/supportable info definition
    "s1_metrics_and_targets": ["27"], # governance oversight of targets
    "s1_materiality": ["17"],         # main materiality principle anchor in appendix B
}

In [29]:
# ── BUILD PARSED PARAGRAPH DICTIONARY ───────────────────────

s1_paragraphs = parse_ifrs_paragraphs(md_text)

parsed_s1 = {
    p.para_id: p
    for p in s1_paragraphs
}

print("Parsed S1 paragraphs:", len(parsed_s1))
print("First 20 paragraph IDs:")
print(list(parsed_s1.keys())[:20])

# Build semantic chunks
s1_semantic_chunks = assemble_chunks(
    parsed=parsed_s1,
    chunk_map=IFRS_S1_CHUNK_MAP,
    anchor_xrefs=ANCHOR_INJECTIONS
)

print("S1 semantic chunks:", len(s1_semantic_chunks))

for c in s1_semantic_chunks:
    print(c.chunk_id, "|", c.display_name, "| chars:", len(c.text), "| paragraphs:", c.metadata["paragraph_ids"])

Parsed S1 paragraphs: 139
First 20 paragraph IDs:
['3', '4', '5', '1', '6', '8', '9', '11', '12', '13', '14', '7', '20', '23', '24', '26', '27', '28', '10', '30']
S1 semantic chunks: 15
s1_objective_scope | Objective and Scope | chars: 956 | paragraphs: ['1', '2', '3', '4', '5', '6', '7', '8', '9']
s1_fair_presentation | Fair Presentation | chars: 478 | paragraphs: ['11', '12', '13', '14', '15', '16']
s1_materiality | Materiality | chars: 8471 | paragraphs: ['17', '18', '19', 'B13', 'B14', 'B15', 'B16', 'B17', 'B18', 'B19', 'B20', 'B21', 'B22', 'B23', 'B24', 'B25', 'B26', 'B27', 'B28', 'B29', 'B30', 'B31', 'B32', 'B33', 'B34', 'B35', 'B36', 'B37']
s1_connected_information | Connected Information | chars: 1546 | paragraphs: ['21', '22', '23', '24', 'B39', 'B40', 'B41', 'B42', 'B43', 'B44']
s1_governance | Governance — Core Content | chars: 158 | paragraphs: ['26', '27']
s1_strategy_risks | Strategy — Risks and Opportunities | chars: 503 | paragraphs: ['28', '29', '30', '31']
s1_strategy

In [30]:
# ── DEBUG PARSED PARAGRAPHS ─────────────────────────────────

import pandas as pd

paragraphs_df = pd.DataFrame([
    {
        "para_id": p.para_id,
        "section": p.section,
        "is_bold": p.is_bold,
        "text_len": len(p.text),
        "word_count": len(p.text.split()),
        "cross_refs": p.raw_cross_refs,
        "text_preview": p.text[:300]
    }
    for p in s1_paragraphs
])

print("Paragraph count:", len(paragraphs_df))
display(paragraphs_df.head(30))

print("\nParagraphs by section:")
display(paragraphs_df["section"].value_counts())

print("\nShortest paragraphs:")
display(paragraphs_df.sort_values("word_count").head(15))

print("\nLongest paragraphs:")
display(paragraphs_df.sort_values("word_count", ascending=False).head(15))

Paragraph count: 164


,para_id,section,is_bold,text_len,word_count,cross_refs,text_preview
0,3,objective,False,102,13,[],IFRS S1 GENERAL REQUIREMENTS FOR DISCLOSURE OF SUSTAINABILITYRELATED FINANCIAL INFORMATION — JUNE 2023
1,4,objective,False,40,4,[],IFRS SUSTAINABILITY DISCLOSURE STANDARDS
2,5,scope,False,105,14,[],## IFRS S1 GENERAL REQUIREMENTS FOR DISCLOSURE OF SUSTAINABILITYRELATED FINANCIAL INFORMATION — JUNE 2023
3,1,objective,False,375,48,[],**The objective of IFRS S1** _**General Requirements for Disclosure of Sustainabilityrelated Financial Information**_ **is to require an entity to disclose information about its sustainability-related risks and opportunities that is useful to** _**primary users of general purpose financial reports**
4,6,scope,False,43,5,[],## IFRS SUSTAINABILITY DISCLOSURE STANDARDS
5,8,scope,False,332,45,[],**An entity may apply IFRS Sustainability Disclosure Standards irrespective of whether the entity** ’ **s related general purpose financial statements (referred to as** ‘ **financial statements** ’ **) are prepared in accordance with IFRS Accounting Standards or other generally accepted accounting p
6,9,scope,False,358,47,[],"This Standard uses terminology suitable for profit-oriented entities, including public-sector business entities. If entities with not-for-profit activities in the private sector or the public sector apply this Standard, they might need to amend the descriptions used for particular items of informati"
7,11,conceptual_foundations,False,211,27,[],**A complete set of sustainability-related financial disclosures shall present fairly all sustainability-related risks and opportunities that could reasonably be expected to affect an entity** ’ **s prospects.**
8,12,conceptual_foundations,False,170,26,[B1 – B12],"To identify sustainability-related risks and opportunities that could reasonably be expected to affect an entity ’ s prospects, an entity shall apply paragraphs B1 – B12."
9,13,conceptual_foundations,False,446,58,[],"**Fair presentation requires disclosure of relevant information about sustainability-related risks and opportunities that could reasonably be expected to affect the entity** ’ **s prospects, and their faithful representation in accordance with the principles set out in this Standard. To achieve fait"



Paragraphs by section:


section
appendix_b                42
core_content              40
appendix_d                23
conceptual_foundations    22
general_requirements      16
scope                      7
judgements_and_errors      7
objective                  3
appendix_c                 3
appendix_e                 1
Name: count, dtype: int64


Shortest paragraphs:


,para_id,section,is_bold,text_len,word_count,cross_refs,text_preview
82,B9,appendix_b,False,3,1,[],B10
66,79,judgements_and_errors,False,2,1,[],80
1,4,objective,False,40,4,[],IFRS SUSTAINABILITY DISCLOSURE STANDARDS
46,59,general_requirements,False,25,4,[],An entity shall identify:
65,78,judgements_and_errors,False,23,4,[],## **An entity shall:**
70,22,conceptual_foundations,False,40,4,[],IFRS SUSTAINABILITY DISCLOSURE STANDARDS
51,18,conceptual_foundations,False,40,4,[],IFRS SUSTAINABILITY DISCLOSURE STANDARDS
4,6,scope,False,43,5,[],## IFRS SUSTAINABILITY DISCLOSURE STANDARDS
35,14,conceptual_foundations,False,43,5,[],## IFRS SUSTAINABILITY DISCLOSURE STANDARDS
62,20,conceptual_foundations,False,43,5,[],## IFRS SUSTAINABILITY DISCLOSURE STANDARDS



Longest paragraphs:


,para_id,section,is_bold,text_len,word_count,cross_refs,text_preview
75,B3,appendix_b,False,1405,233,[],"For example, if an entity ’ s business model depends on a natural resource — such as water — the entity could both affect and be affected by the quality, availability and affordability of that resource. Specifically, degradation or depletion of that resource — including resulting from the entity ’ s"
74,B2,appendix_b,False,1082,167,[],"An entity ’ s sustainability-related risks and opportunities arise out of the interactions between the entity and its stakeholders, society, the economy and the natural environment throughout the entity ’ s value chain. These interactions — which can be direct and indirect — result from operating an"
110,B43,appendix_b,False,921,144,[],"For example, in providing connected information an entity might need to explain the effect or likely effect of its strategy on its financial statements and financial planning, or explain how that strategy relates to the metrics the entity uses to measure progress against targets. Another entity migh"
92,B23,appendix_b,False,934,143,[],"When considering possible outcomes, an entity shall consider all pertinent facts and circumstances. Information about a possible future event is more likely to be judged as being material if the potential effects are significant and the event is likely to occur. However, an entity shall also conside"
131,D4,appendix_d,False,909,133,[],Relevant sustainability-related financial information is capable of making a difference in the decisions made by primary users. Information may be capable of making a difference in a decision even if some users choose not to take advantage of it or are already aware of it from other sources. Sustain
88,B19,appendix_b,False,890,122,[57 58],"Materiality judgements are specific to an entity. Consequently, this Standard does not specify any thresholds for materiality or predetermine what would be material in a particular situation. B20 To identify material information about a sustainability-related risk or opportunity, an entity shall app"
48,61,general_requirements,False,722,118,[],"Subject to any regulation or other requirements that apply to an entity, there are various possible locations in its general purpose financial reports in which to disclose sustainability-related financial information. Sustainability-related financial disclosures could be included in an entity ’ s ma"
93,B24,appendix_b,False,710,115,[],"If a possible future event is expected to affect an entity ’ s cash flows, but only many years in the future, information about that event is usually less likely to be judged material than information about a possible future event with similar effects that are expected to occur sooner. However, in s"
77,B5,appendix_b,False,705,114,[],"An entity ’ s dependencies and impacts are not limited to resources the entity engages with directly, and to the entity ’ s direct relationships. Those dependencies and impacts also relate to resources and relationships throughout the entity ’ s value chain. For example, they can relate to the entit"
101,B30,appendix_b,False,685,98,[],An entity shall not aggregate information if doing so would obscure information that is material. Information shall be aggregated if items of information have shared characteristics and shall not be aggregated if they do not have shared characteristics. The entity might need to disaggregate informat


In [31]:
# ── DEBUG IMPORTANT IFRS S1 PARAGRAPHS ──────────────────────

important_s1_ids = [
    "1", "2", "3", "4",
    "11", "12", "13", "14", "15", "16",
    "17", "18", "19",
    "21", "22", "23", "24",
    "26", "27",
    "28", "29", "30", "31",
    "34", "35", "36", "37", "38", "39", "40",
    "43", "44",
    "45", "46", "47", "48", "49", "50", "51", "52", "53",
    "77", "78", "79", "80", "81", "82",
    "83", "84", "85", "86",
    "B13", "B14", "B15", "B34", "B49", "B50", "B54",
    "D1", "D2", "D16"
]

missing = []
for pid in important_s1_ids:
    if pid not in parsed_s1:
        missing.append(pid)

print("Missing important paragraph IDs:", missing)

for pid in important_s1_ids:
    if pid in parsed_s1:
        p = parsed_s1[pid]
        print("=" * 100)
        print(f"§{pid} | section={p.section} | bold={p.is_bold} | words={len(p.text.split())}")
        print(p.text[:800])

Missing important paragraph IDs: ['2', '48', '49', '51', '52', '53', '80', '82', '84', '85', '86', 'B15', 'D1', 'D2']
§1 | section=objective | bold=False | words=48
**The objective of IFRS S1** _**General Requirements for Disclosure of Sustainabilityrelated Financial Information**_ **is to require an entity to disclose information about its sustainability-related risks and opportunities that is useful to** _**primary users of general purpose financial reports**_ **in making decisions relating to providing resources to the entity.**[1]
§3 | section=objective | bold=False | words=13
IFRS S1 GENERAL REQUIREMENTS FOR DISCLOSURE OF SUSTAINABILITYRELATED FINANCIAL INFORMATION — JUNE 2023
§4 | section=objective | bold=False | words=4
IFRS SUSTAINABILITY DISCLOSURE STANDARDS
§11 | section=conceptual_foundations | bold=False | words=14
## IFRS S1 GENERAL REQUIREMENTS FOR DISCLOSURE OF SUSTAINABILITYRELATED FINANCIAL INFORMATION — JUNE 2023
§12 | section=conceptual_foundations | bold=False | wor

In [32]:
# ============================================================
# 4. PARAGRAPH PARSER
# ============================================================

# Matches paragraph IDs at beginning of line:
# 1, 27, 51, B13, C2, D16, E4
PARA_START_PATTERN = re.compile(
    r"^(?P<bold_open>\*\*)?(?P<para_id>B\d+|C\d+|D\d+|E\d+|\d+)(?P<bold_close>\*\*)?\s+(?P<body>.+)"
)

XREF_PATTERN = re.compile(
    r"paragraph[s]?\s+([\w\d\(\)–\-,\s]+)",
    re.IGNORECASE
)


def classify_section(para_id: str) -> str:
    """
    Classify IFRS S1 paragraph by broad location.
    """
    if para_id.startswith("B"):
        return "appendix_b"
    if para_id.startswith("C"):
        return "appendix_c"
    if para_id.startswith("D"):
        return "appendix_d"
    if para_id.startswith("E"):
        return "appendix_e"

    num = int(para_id)

    if num <= 4:
        return "objective"
    if num <= 9:
        return "scope"
    if num <= 24:
        return "conceptual_foundations"
    if num <= 53:
        return "core_content"
    if num <= 73:
        return "general_requirements"

    return "judgements_and_errors"


def looks_like_new_paragraph(line: str) -> bool:
    """
    True if a line starts with a valid IFRS paragraph ID.
    """
    return bool(PARA_START_PATTERN.match(line.strip()))


def is_noise_text(text: str) -> bool:
    """
    Remove fake paragraphs caused by headers/footers.
    """
    t = str(text).strip()
    tl = t.lower()

    if len(t.split()) < 5:
        return True

    noise_patterns = [
        r"^#+\s*ifrs s1 general requirements",
        r"^#+\s*ifrs sustainability disclosure standards",
        r"^ifrs s1 general requirements",
        r"^ifrs sustainability disclosure standards$",
        r"^ifrs sustainability disclosure standards",
    ]

    if any(re.search(p, tl, re.IGNORECASE) for p in noise_patterns):
        return True

    noise_keywords = [
        "copyright",
        "all rights reserved",
        "reproduction and use rights",
        "disclaimer",
        "isbn",
        "intentionally omitted",
    ]

    if any(k in tl for k in noise_keywords):
        return True

    return False


def parse_ifrs_paragraphs(md_text: str) -> list[IFRSParagraph]:
    """
    Line-aware parser:
    - detects paragraph starts
    - attaches following continuation lines and bullets to same paragraph
    - does not invent paragraph IDs
    """

    paragraphs = []

    current_para_id = None
    current_lines = []
    current_is_bold = False

    lines = md_text.splitlines()

    def flush_current():
        nonlocal current_para_id, current_lines, current_is_bold

        if current_para_id is None or not current_lines:
            return

        full_text = "\n".join(current_lines).strip()
        full_text = re.sub(r"[ \t]+", " ", full_text)
        full_text = re.sub(r"\n{2,}", "\n", full_text).strip()

        if is_noise_text(full_text):
            current_para_id = None
            current_lines = []
            current_is_bold = False
            return

        cross_refs = XREF_PATTERN.findall(full_text)

        paragraphs.append(IFRSParagraph(
            para_id=current_para_id,
            section=classify_section(current_para_id),
            is_bold=current_is_bold,
            text=full_text,
            raw_cross_refs=cross_refs,
        ))

        current_para_id = None
        current_lines = []
        current_is_bold = False

    for line in lines:
        stripped = line.strip()

        if not stripped:
            # Keep blank line inside paragraph only if paragraph exists
            if current_para_id is not None:
                current_lines.append("")
            continue

        match = PARA_START_PATTERN.match(stripped)

        if match:
            # Save previous paragraph
            flush_current()

            current_para_id = match.group("para_id")
            current_is_bold = bool(match.group("bold_open"))

            body = match.group("body").strip()

            # Remove closing bold marker at end if exists
            body = body.replace("**", "")

            current_lines = [body]

        else:
            # Continuation line / bullet / sub-item
            if current_para_id is not None:
                current_lines.append(stripped)

    # Save final paragraph
    flush_current()

    return paragraphs

In [33]:
# ============================================================
# 5. DEDUPLICATION / BEST PARAGRAPH SELECTION
# ============================================================

def para_quality_score(p: IFRSParagraph) -> int:
    """
    Pick the best version when the same paragraph ID appears multiple times.
    Rewards actual IFRS disclosure language.
    Penalizes headers and short fragments.
    """

    text = p.text
    tl = text.lower()
    words = text.split()

    score = len(words)

    if is_noise_text(text):
        score -= 1000

    if len(words) < 8:
        score -= 200

    useful_terms = [
        "shall disclose",
        "shall",
        "sustainability-related risks and opportunities",
        "general purpose financial reports",
        "users of general purpose financial reports",
        "governance",
        "strategy",
        "risk management",
        "metrics",
        "targets",
        "financial position",
        "financial performance",
        "cash flows",
        "material information",
    ]

    for term in useful_terms:
        if term in tl:
            score += 50

    if p.is_bold:
        score += 20

    return score


s1_paragraphs_raw = parse_ifrs_paragraphs(md_text)

best_by_id = {}

for p in s1_paragraphs_raw:
    current = best_by_id.get(p.para_id)

    if current is None:
        best_by_id[p.para_id] = p
    else:
        if para_quality_score(p) > para_quality_score(current):
            best_by_id[p.para_id] = p

parsed_s1 = best_by_id

print("Raw parsed paragraphs:", len(s1_paragraphs_raw))
print("Unique paragraph IDs:", len(parsed_s1))
print("First 40 IDs:", list(parsed_s1.keys())[:40])

Raw parsed paragraphs: 102
Unique paragraph IDs: 102
First 40 IDs: ['1', '8', '9', '11', '12', '13', '14', '20', '23', '24', '26', '27', '28', '30', '31', '32', '33', '34', '36', '37', '40', '41', '42', '43', '45', '46', '47', '50', '56', '57', '60', '61', '62', '63', '65', '66', '68', '72', '74', '76']


In [34]:
# ============================================================
# 6. SEMANTIC CHUNK MAP — IFRS S1
# ============================================================

IFRS_S1_CHUNK_MAP = [
    (
        "s1_objective_scope",
        "Objective and Scope",
        list(range(1, 10)),
        [],
        [],
    ),
    (
        "s1_fair_presentation",
        "Fair Presentation",
        list(range(11, 17)),
        [],
        [],
    ),
    (
        "s1_materiality",
        "Materiality",
        list(range(17, 20)),
        [f"B{i}" for i in range(13, 38)],
        [],
    ),
    (
        "s1_connected_information",
        "Connected Information",
        list(range(21, 25)),
        [f"B{i}" for i in range(39, 45)],
        [],
    ),
    (
        "s1_governance",
        "Governance — Core Content",
        [26, 27],
        [],
        ["IFRS_S2_5", "IFRS_S2_6", "IFRS_S2_7"],
    ),
    (
        "s1_strategy_risks",
        "Strategy — Risks and Opportunities",
        list(range(28, 32)),
        [],
        [],
    ),
    (
        "s1_strategy_financial_effects",
        "Strategy — Financial Effects",
        list(range(34, 41)),
        [],
        [],
    ),
    (
        "s1_risk_management",
        "Risk Management",
        [43, 44],
        [],
        [],
    ),
    (
        "s1_metrics_and_targets",
        "Metrics and Targets",
        list(range(45, 54)),
        [f"B{i}" for i in range(49, 55)],
        [],
    ),
    (
        "s1_sources_of_guidance",
        "Sources of Guidance",
        list(range(54, 60)),
        [],
        [],
    ),
    (
        "s1_measurement_uncertainty",
        "Measurement Uncertainty",
        list(range(77, 83)),
        [],
        [],
    ),
    (
        "s1_errors",
        "Errors",
        list(range(83, 87)),
        [f"B{i}" for i in range(55, 60)],
        [],
    ),
    (
        "s1_qualitative_characteristics_fundamental",
        "Appendix D — Fundamental Qualitative Characteristics",
        [f"D{i}" for i in range(1, 16)],
        [],
        [],
    ),
    (
        "s1_qualitative_characteristics_enhancing",
        "Appendix D — Enhancing Qualitative Characteristics",
        [f"D{i}" for i in range(16, 34)],
        [],
        [],
    ),
]


def infer_pillar(chunk_id: str) -> str:
    mapping = {
        "governance": "governance",
        "strategy": "strategy",
        "risk_management": "risk_management",
        "metrics": "metrics_and_targets",
        "materiality": "general_requirements",
        "sources": "general_requirements",
        "uncertainty": "general_requirements",
        "errors": "general_requirements",
    }

    for key, pillar in mapping.items():
        if key in chunk_id:
            return pillar

    return "other"


def build_anchor(para: IFRSParagraph) -> str:
    short = para.text[:250].rsplit(" ", 1)[0]
    return f"[Anchor §{para.para_id}]: {short}..."


ANCHOR_INJECTIONS = {
    "s1_governance": ["51"],
    "s1_strategy_risks": ["B6"],
    "s1_metrics_and_targets": ["27"],
    "s1_materiality": ["17"],
}


def assemble_chunks(
    parsed: dict[str, IFRSParagraph],
    chunk_map: list,
    anchor_xrefs: dict[str, list[str]],
    standard_name: str = "IFRS S1"
) -> list[IFRSChunk]:

    chunks = []

    for chunk_id, display_name, para_ids, appendix_ids, cross_std_refs in chunk_map:

        all_ids = [str(p) for p in para_ids] + [str(p) for p in appendix_ids]
        parts = []
        missing_ids = []

        for pid in all_ids:
            if pid in parsed:
                p = parsed[pid]
                bold_marker = "**" if p.is_bold else ""
                parts.append(f"{bold_marker}§{p.para_id} {p.text}")
            else:
                missing_ids.append(pid)

        # Inject short anchors
        injected_anchors = []
        for anchor_para_id in anchor_xrefs.get(chunk_id, []):
            if anchor_para_id in parsed:
                parts.append(build_anchor(parsed[anchor_para_id]))
                injected_anchors.append(anchor_para_id)

        full_text = "\n\n".join(parts).strip()

        all_xrefs = []
        for pid in all_ids:
            if pid in parsed:
                all_xrefs.extend(parsed[pid].raw_cross_refs)

        chunks.append(IFRSChunk(
            chunk_id=chunk_id,
            display_name=display_name,
            text=full_text,
            metadata={
                "standard": standard_name,
                "paragraph_ids": all_ids,
                "missing_paragraph_ids": missing_ids,
                "cross_std_refs": cross_std_refs,
                "injected_anchors": injected_anchors,
                "internal_xrefs": list(set(all_xrefs)),
                "has_bold_principle": any(
                    parsed[pid].is_bold
                    for pid in all_ids
                    if pid in parsed
                ),
                "pillar": infer_pillar(chunk_id),
                "word_count": len(full_text.split()),
            }
        ))

    return chunks


s1_semantic_chunks = assemble_chunks(
    parsed=parsed_s1,
    chunk_map=IFRS_S1_CHUNK_MAP,
    anchor_xrefs=ANCHOR_INJECTIONS,
    standard_name="IFRS S1"
)

print("S1 semantic chunks:", len(s1_semantic_chunks))

for c in s1_semantic_chunks:
    print(
        c.chunk_id,
        "| words:", len(c.text.split()),
        "| missing:", c.metadata["missing_paragraph_ids"]
    )

S1 semantic chunks: 14
s1_objective_scope | words: 548 | missing: ['2', '3', '4', '5', '6', '7']
s1_fair_presentation | words: 381 | missing: ['15', '16']
s1_materiality | words: 2257 | missing: ['17', '18', '19', 'B15', 'B16', 'B17', 'B18', 'B20', 'B21', 'B30', 'B31', 'B35', 'B36', 'B37']
s1_connected_information | words: 664 | missing: ['21', '22', 'B39', 'B40', 'B41', 'B44']
s1_governance | words: 357 | missing: []
s1_strategy_risks | words: 473 | missing: ['29']
s1_strategy_financial_effects | words: 626 | missing: ['35', '38', '39']
s1_risk_management | words: 285 | missing: ['44']
s1_metrics_and_targets | words: 1188 | missing: ['48', '49', '51', '52', '53']
s1_sources_of_guidance | words: 392 | missing: ['54', '55', '58', '59']
s1_measurement_uncertainty | words: 457 | missing: ['78', '79', '80', '81', '82']
s1_errors | words: 986 | missing: ['84', '85', '86', 'B55', 'B57', 'B59']
s1_qualitative_characteristics_fundamental | words: 799 | missing: ['D1', 'D2', 'D3', 'D5', 'D7', '

In [35]:
# ============================================================
# 7. DEBUG PARAGRAPH QUALITY
# ============================================================

paragraphs_df = pd.DataFrame([
    {
        "para_id": p.para_id,
        "section": p.section,
        "is_bold": p.is_bold,
        "word_count": len(p.text.split()),
        "quality_score": para_quality_score(p),
        "text_preview": p.text[:400],
    }
    for p in parsed_s1.values()
])

display(paragraphs_df.sort_values("para_id").head(50))

print("Paragraphs by section:")
display(paragraphs_df["section"].value_counts())

print("Shortest paragraphs:")
display(paragraphs_df.sort_values("word_count").head(20))

,para_id,section,is_bold,word_count,quality_score,text_preview
0,1,objective,False,386,636,The objective of IFRS S1 _General Requirements for Disclosure of Sustainabilityrelated Financial Information_ is to require an entity to disclose information about its sustainability-related risks and opportunities that is useful to _primary users of general purpose financial reports_ in making decisions relating to providing resources to the entity.[1]\n- 2 Information about sustainability-related
3,11,conceptual_foundations,False,27,127,A complete set of sustainability-related financial disclosures shall present fairly all sustainability-related risks and opportunities that could reasonably be expected to affect an entity ’ s prospects.
4,12,conceptual_foundations,False,26,126,"To identify sustainability-related risks and opportunities that could reasonably be expected to affect an entity ’ s prospects, an entity shall apply paragraphs B1 – B12."
5,13,conceptual_foundations,False,58,158,"Fair presentation requires disclosure of relevant information about sustainability-related risks and opportunities that could reasonably be expected to affect the entity ’ s prospects, and their faithful representation in accordance with the principles set out in this Standard. To achieve faithful representation, an entity shall provide a complete, neutral and accurate depiction of those sustainab"
6,14,conceptual_foundations,False,266,616,"Materiality is an entity-specific aspect of relevance based on the nature or magnitude, or both, of the items to which the information relates, in the context of the entity ’ s sustainability-related financial disclosures.\n- 15\nFair presentation also requires an entity:\n- (a) to disclose information that is comparable, verifiable, timely and understandable; and\n- (b) to disclose additional informa"
7,20,conceptual_foundations,False,174,574,An entity ’ s sustainability-related financial disclosures shall be for the same reporting entity as the related financial statements (see paragraph B38).\n## **Connected information**\n- 21 **An entity shall provide information in a manner that enables users of general purpose financial reports to understand the following types of connections:**\n- **(a) the connections between the items to which th
8,23,conceptual_foundations,False,46,96,Data and assumptions used in preparing the sustainability-related financial disclosures shall be consistent — to the extent possible considering the — requirements of IFRS Accounting Standards or other applicable GAAP with the corresponding data and assumptions used in preparing the related financial statements (see paragraph B42).
9,24,conceptual_foundations,False,170,520,"When currency is specified as the unit of measure in the sustainabilityrelated financial disclosures, the entity shall use the presentation currency of its related financial statements.\n## **Core content**\n- 25 **Unless another IFRS Sustainability Disclosure Standard permits or requires otherwise in specified circumstances, an entity shall provide disclosures about:**\n- **(a) governance** — **the"
10,26,core_content,False,37,237,"The objective of sustainability-related financial disclosures on governance is to enable users of general purpose financial reports to understand the governance processes, controls and procedures an entity uses to monitor, manage and oversee sustainability-related risks and opportunities."
11,27,core_content,False,318,718,"To achieve this objective, an entity shall disclose information about:\n- (a) the governance body(s) (which can include a board, committee or equivalent body charged with governance) or individual(s) responsible for oversight of sustainability-related risks and opportunities. Specifically, the entity shall identify that body(s) or individual(s) and disclose information about:\n- (i) how responsibili"


Paragraphs by section:


section
appendix_b                36
appendix_d                21
core_content              18
general_requirements      10
conceptual_foundations     7
judgements_and_errors      4
appendix_c                 3
scope                      2
objective                  1
Name: count, dtype: int64

Shortest paragraphs:


,para_id,section,is_bold,word_count,quality_score,text_preview
89,D18,appendix_d,False,12,62,Sustainability-related financial disclosures shall be provided in a way that enhances comparability.
18,36,core_content,False,14,14,"In providing quantitative information, an entity may disclose a single amount or a range."
87,D16,appendix_d,False,18,18,"The usefulness of sustainability-related financial information is enhanced if it is comparable, verifiable, timely and understandable.\n## **Comparability**"
30,60,general_requirements,False,21,71,An entity is required to provide disclosures required by IFRS Sustainability Disclosure Standards as part of its general purpose financial reports.
62,B32,appendix_b,False,21,121,"An entity shall disclose material sustainability-related financial information, even if law or regulation permits the entity not to disclose such information."
85,D11,appendix_d,False,23,73,A complete depiction of a sustainability-related risk or opportunity includes all material information necessary for primary users to understand that risk or opportunity.
70,B49,appendix_b,False,25,75,Paragraph 70 requires an entity to disclose comparative information in respect of the preceding period for all amounts disclosed in the reporting period.\n## **Metrics**
4,12,conceptual_foundations,False,26,126,"To identify sustainability-related risks and opportunities that could reasonably be expected to affect an entity ’ s prospects, an entity shall apply paragraphs B1 – B12."
3,11,conceptual_foundations,False,27,127,A complete set of sustainability-related financial disclosures shall present fairly all sustainability-related risks and opportunities that could reasonably be expected to affect an entity ’ s prospects.
74,B53,appendix_b,False,27,127,"If an entity introduces a new metric in the reporting period, it shall disclose a comparative amount for that metric unless it is impracticable to do so."


In [36]:
# ============================================================
# 10. SHOW SPECIFIC CHUNKS
# ============================================================

def show_chunk(chunk_id: str, max_chars: int = 5000):
    matches = [c for c in s1_semantic_chunks if c.chunk_id == chunk_id]

    if not matches:
        print("Missing chunk:", chunk_id)
        return

    c = matches[0]
    print("=" * 120)
    print("chunk_id:", c.chunk_id)
    print("display_name:", c.display_name)
    print("pillar:", c.metadata["pillar"])
    print("word_count:", len(c.text.split()))
    print("paragraph_ids:", c.metadata["paragraph_ids"])
    print("missing_paragraph_ids:", c.metadata["missing_paragraph_ids"])
    print("-" * 120)
    print(c.text[:max_chars])


show_chunk("s1_governance")
show_chunk("s1_strategy_risks")
show_chunk("s1_strategy_financial_effects")
show_chunk("s1_risk_management")
show_chunk("s1_metrics_and_targets")

chunk_id: s1_governance
display_name: Governance — Core Content
pillar: governance
word_count: 357
paragraph_ids: ['26', '27']
missing_paragraph_ids: []
------------------------------------------------------------------------------------------------------------------------
§26 The objective of sustainability-related financial disclosures on governance is to enable users of general purpose financial reports to understand the governance processes, controls and procedures an entity uses to monitor, manage and oversee sustainability-related risks and opportunities.

§27 To achieve this objective, an entity shall disclose information about:
- (a) the governance body(s) (which can include a board, committee or equivalent body charged with governance) or individual(s) responsible for oversight of sustainability-related risks and opportunities. Specifically, the entity shall identify that body(s) or individual(s) and disclose information about:
- (i) how responsibilities for sustainability-rel